# ETL 파이프라인 구축

데이터 추출, 변환, 로드 파이프라인을 구축합니다.

## 학습 목표
1. ETL 프로세스 이해
2. 데이터 검증
3. 특성 공학
4. 파이프라인 자동화

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import logging
from datetime import datetime
import json

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. 샘플 데이터 생성

In [ ]:
# 샘플 데이터 생성
np.random.seed(42)
n_samples = 1000

data = {
    'user_id': range(1, n_samples + 1),
    'age': np.random.randint(18, 70, n_samples),
    'gender': np.random.choice(['M', 'F', None], n_samples, p=[0.48, 0.48, 0.04]),
    'income': np.random.normal(50000, 20000, n_samples),
    'purchase_amount': np.random.exponential(100, n_samples),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_samples),
    'signup_date': pd.date_range('2020-01-01', periods=n_samples, freq='D'),
    'is_premium': np.random.choice([True, False], n_samples, p=[0.3, 0.7])
}

# 일부 결측값 추가
df = pd.DataFrame(data)
df.loc[np.random.choice(df.index, 50), 'income'] = np.nan
df.loc[np.random.choice(df.index, 30), 'age'] = np.nan

print(f"데이터 shape: {df.shape}")
df.head()

## 2. ETL 클래스 정의

In [ ]:
class DataPipeline:
    def __init__(self, config=None):
        self.config = config or {}
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.stats = {}
    
    def extract(self, source):
        """데이터 추출"""
        logger.info(f"Extracting data from {type(source).__name__}")
        
        if isinstance(source, pd.DataFrame):
            df = source.copy()
        elif isinstance(source, str):
            if source.endswith('.csv'):
                df = pd.read_csv(source)
            elif source.endswith('.json'):
                df = pd.read_json(source)
            else:
                raise ValueError(f"Unsupported file format: {source}")
        else:
            raise TypeError(f"Unsupported source type: {type(source)}")
        
        self.stats['raw_rows'] = len(df)
        logger.info(f"Extracted {len(df)} rows")
        return df
    
    def validate(self, df):
        """데이터 검증"""
        logger.info("Validating data...")
        
        issues = []
        
        # 결측값 체크
        missing = df.isnull().sum()
        for col, count in missing.items():
            if count > 0:
                pct = count / len(df) * 100
                issues.append(f"{col}: {count} missing ({pct:.1f}%)")
        
        # 중복 체크
        duplicates = df.duplicated().sum()
        if duplicates > 0:
            issues.append(f"Duplicate rows: {duplicates}")
        
        self.stats['validation_issues'] = issues
        
        if issues:
            logger.warning(f"Validation issues found: {len(issues)}")
            for issue in issues:
                logger.warning(f"  - {issue}")
        else:
            logger.info("No validation issues found")
        
        return df, issues
    
    def transform(self, df):
        """데이터 변환"""
        logger.info("Transforming data...")
        df = df.copy()
        
        # 1. 결측값 처리
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].isnull().any():
                median = df[col].median()
                df[col].fillna(median, inplace=True)
                logger.info(f"  Filled {col} missing with median: {median:.2f}")
        
        categorical_cols = df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df[col].isnull().any():
                mode = df[col].mode()[0]
                df[col].fillna(mode, inplace=True)
                logger.info(f"  Filled {col} missing with mode: {mode}")
        
        # 2. 이상치 처리 (IQR method)
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            if outliers > 0:
                df[col] = df[col].clip(lower, upper)
                logger.info(f"  Clipped {outliers} outliers in {col}")
        
        # 3. 특성 공학
        if 'signup_date' in df.columns:
            df['signup_year'] = pd.to_datetime(df['signup_date']).dt.year
            df['signup_month'] = pd.to_datetime(df['signup_date']).dt.month
            df['days_since_signup'] = (pd.Timestamp.now() - pd.to_datetime(df['signup_date'])).dt.days
            logger.info("  Created date features")
        
        if 'age' in df.columns:
            df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 100], 
                                     labels=['Young', 'Adult', 'Middle', 'Senior'])
            logger.info("  Created age_group feature")
        
        # 4. 인코딩
        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
                self.label_encoders[col] = le
        
        self.stats['transformed_rows'] = len(df)
        self.stats['final_columns'] = list(df.columns)
        
        logger.info(f"Transformation complete. Shape: {df.shape}")
        return df
    
    def load(self, df, destination):
        """데이터 로드"""
        logger.info(f"Loading data to {destination}...")
        
        if destination.endswith('.csv'):
            df.to_csv(destination, index=False)
        elif destination.endswith('.parquet'):
            df.to_parquet(destination, index=False)
        elif destination.endswith('.json'):
            df.to_json(destination, orient='records')
        else:
            raise ValueError(f"Unsupported destination format: {destination}")
        
        logger.info(f"Saved {len(df)} rows to {destination}")
        return destination
    
    def run(self, source, destination):
        """전체 ETL 파이프라인 실행"""
        start_time = datetime.now()
        logger.info(f"Starting ETL pipeline at {start_time}")
        
        # Extract
        df = self.extract(source)
        
        # Validate
        df, issues = self.validate(df)
        
        # Transform
        df = self.transform(df)
        
        # Load
        self.load(df, destination)
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        self.stats['duration_seconds'] = duration
        
        logger.info(f"ETL pipeline completed in {duration:.2f} seconds")
        return df, self.stats

## 3. 파이프라인 실행

In [ ]:
# 파이프라인 실행
pipeline = DataPipeline()
transformed_df, stats = pipeline.run(df, 'transformed_data.csv')

print("\n" + "="*50)
print("ETL 통계:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# 변환된 데이터 확인
print(f"\n변환된 데이터 shape: {transformed_df.shape}")
print(f"\n컬럼: {list(transformed_df.columns)}")
transformed_df.head()

## 4. 데이터 품질 리포트

In [ ]:
def generate_quality_report(df):
    """데이터 품질 리포트 생성"""
    report = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'memory_usage_mb': df.memory_usage(deep=True).sum() / 1024**2,
        'missing_values': df.isnull().sum().to_dict(),
        'data_types': df.dtypes.astype(str).to_dict(),
        'numeric_stats': df.describe().to_dict(),
    }
    return report

quality_report = generate_quality_report(transformed_df)
print("데이터 품질 리포트:")
print(f"  총 행: {quality_report['total_rows']}")
print(f"  총 열: {quality_report['total_columns']}")
print(f"  메모리 사용량: {quality_report['memory_usage_mb']:.2f} MB")

## 연습 문제

1. Apache Airflow DAG로 이 파이프라인을 스케줄링해보세요.
2. Great Expectations로 데이터 검증을 추가해보세요.
3. 파이프라인에 로깅과 알림 기능을 추가해보세요.
4. 증분 처리(Incremental Processing)를 구현해보세요.